In [ ]:
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

# packages loader
use_package <- function(pkg) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg)
    library(pkg, character.only = TRUE)
  }
}

use_package("MASS")
use_package("readxl") # not working
use_package("gdata") # not working
use_package("FKSUM")


Helper functions

In [ ]:
# claw 2
rclaw = function(n) {
    x = c(0,1,2,3,4,5)
    y = sample(x,n,replace=TRUE,c(0.1,0.1,0.1,0.1,0.1,0.5)) 
    z = rnorm(n,0,1)
    z1 = ifelse(y<5,(y/2)-1+0.1*z,z)
    return(z1)
}

# z<-rnorm(50,0,1)
# > z1<-ifelse(y<5,(y/2)-1+0.1*z,z)

# function for analysis densities 
analyze_densities = function(data , kernel){
    ## Densities
    # kernel densities - windows width
    bw_ucv <- ucv(data)
    bw_bcv <- bcv(data)
    bw_sj  <- width.SJ(data)
    bw_dpi <- width.SJ(data, method = "dpi")
    # kernel densities - pdfs
    d_ucv <- density(data, kernel=kernel, bw = bw_ucv)
    d_bcv <- density(data, kernel=kernel, bw = bw_bcv)
    d_sj  <- density(data, kernel=kernel, bw = bw_sj)
    d_dpi <- density(data, kernel=kernel,  bw = bw_dpi)

    ## plots
    # histogram
    hist.FD(data, prob=TRUE, main = "Histogram + KDE", xlab = "Duration (min)")
    # densities - "density" function uses by default gaussian kernel 
    lines(d_ucv, col = "blue", lwd = 2)
    lines(d_bcv, col = "green", lwd = 2)
    lines(d_sj,  col = "red", lwd = 2)
    lines(d_dpi, col = "purple", lwd = 2)
    # legend
    legend("topright", legend = c("UCV", "BCV", "SJ", "DPI"), col = c("blue", "green", "red", "purple"), lwd = 2)
    
    ## Windows results 
    return(data.frame(
        bw_ucv = bw_ucv,
        bw_bcv = bw_bcv,
        bw_sj  = bw_sj,
        bw_dpi = bw_dpi
    ))
}

# Exercise 1
## About methods
- about methods - how they work
## About data
Geyser data set
- describe eruption duration ($duration (min)) and and time between eruptions ($waiting (?)) from (1985-1990)
- 
## My interpretation 
### Densities
#### Methodology
#### Resutls 
### Ratio
#### Methodology
- I skip *BCV* method because it was to *flat*. 
#### Results
-- 
porównac gładkość i rozdzielczość - z czego wynika, że pewne estymacje zwiększają przyrost a inne nie  

In [ ]:
for (kernel in c("cosine", "epanechnikov")){
    analyze_densities(data = data_1,kernel = kernel)
}
# Here we can see that kernels HAVE impact on tails. 

In [ ]:
# parameters
kernel = "epanechnikov"

# methods
data_1 = geyser$duration
ucv(data_1)
bcv(data_1)
width.SJ(data_1)
width.SJ(data_1, method="dpi")

## Densities
# kernel densities - windows width
bw_ucv <- ucv(data_1)
bw_bcv <- bcv(data_1)
bw_sj  <- width.SJ(data_1)
bw_dpi <- width.SJ(data_1, method = "dpi")
# kernel densities - pdfs
d_ucv <- density(data_1, kernel = kernel, bw = bw_ucv)
d_bcv <- density(data_1, kernel = kernel, bw = bw_bcv)
d_sj  <- density(data_1, kernel = kernel, bw = bw_sj)
d_dpi <- density(data_1, kernel = kernel, bw = bw_dpi)


## plots
# histogram
hist.scott(data_1,prob = TRUE, col = "lightgray", main = "Histogram + KDE", xlab = "Duration (min)")
# hist.scott(data_1, prob=TRUE, main = "Histogram + KDE", xlab = "Duration (min)")
# densities - "density" function uses by default gaussian kernel 
lines(d_ucv, col = "blue", lwd = 2)
lines(d_bcv, col = "green", lwd = 2)
lines(d_sj,  col = "red", lwd = 2)
lines(d_dpi, col = "purple", lwd = 2)
# legend
legend("topright", legend = c("UCV", "BCV", "SJ", "DPI"), col = c("blue", "green", "red", "purple"), lwd = 2)

## Maximum comparison
# local maximum function
local_max <- function(pdf, domain) {
  range <- which(pdf$x >= domain[1] & pdf$x <= domain[2])
  max(pdf$y[range])
}
# Dataframe
results <- data.frame(
  Method = c("UCV", "SJ", "DPI"),
  Left   = c(local_max(d_ucv, c(1,3)),
             local_max(d_sj,  c(1,3)),
             local_max(d_dpi, c(1,3))),
  Right  = c(local_max(d_ucv, c(3,5)),
             local_max(d_sj,  c(3,5)),
             local_max(d_dpi, c(3,5)))
)
results$Ratio = results$Left / results$Right
print(results)

# Exercise 2

In [ ]:
# Results
n_vector = c(100, 200, 300, 1000, 5000, 10000, 15000, 20000)
# n_vector = c(1000)
df = data.frame()
# analyze
for (n in n_vector){
    probe = rclaw(n)
    loc_df = analyze_densities(probe, "gaussian") 
    # adding index column (n)
    loc_df$n = n
    # adding to global df
    df = rbind(df, loc_df)
}
# setting columns right
df = df[, c("n", "bw_ucv", "bw_bcv", "bw_sj", "bw_dpi")]

print(df)

# Exercise 3 

In [ ]:
data_3 = read_excel("Hidalgo1872.xls")

print(analyze_densities(data_3, "rectangular"))


# Exercise 4 

In [ ]:
data_4 <- read.table("ushighways.txt", header = TRUE, sep = " ")
print(analyze_densities(data_4$Approx.Miles, "gaussian"))

# Exercise 5 

In [ ]:
# part 1 - fk_sum
set.seed(1)
n <- 5000
x <- rbeta(n, 2, 2) * 10
fx <- 3 * sin(2 * x) + 10 * (x > 5) * (x - 5)
y <- fx + rt(n, 3) + (rgamma(n, 2, 2) - 1) * ((x - 5)^2 + 3)
xeval <- seq(0, 10, length.out = 1000)

par(mfrow = c(1, 2))
for (h in c(0.25, 0.05)) {
  plot(x, y, xlab = "x", ylab = "y", main = paste("h =", h))
  
  # Kernel regression estimate
  fhat <- fk_sum(x, y, h, x_eval = xeval) / fk_sum(x, rep(1, n), h, x_eval = xeval)
  
  # Estimated function
  lines(xeval, fhat, col = 2, lwd = 2)
  
  # True function
  lines(xeval, 3 * sin(2 * xeval) + 10 * (xeval > 5) * (xeval - 5), col = 4, lwd = 2)
  legend("topright", legend = c("Estymacja", "Prawdziwa f(x)"), col = c(2, 4), lwd = 2)
}


# part 2 użycie fk_regression
par(mfrow = c(1, 2))

# Domyślna szerokość jądra
plot(fk_regression(x, y, type = "NW"), main = "NW: domyślne h")
lines(xeval, 3 * sin(2 * xeval) + 10 * (xeval > 5) * (xeval - 5), col = 3, lwd = 2)

# Automatyczny wybór h przez cross-validation
plot(fk_regression(x, y, type = "NW", h = "cv"), main = "NW: h = cv")
lines(xeval, 3 * sin(2 * xeval) + 10 * (xeval > 5) * (xeval - 5), col = 3, lwd = 2)


# Exercise 6